## Install + imports

In [ ]:
!pip install requests tqdm pandas

In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm
import os

## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##  Folder Setup

In [ ]:
BASE_DIR = "/content/drive/MyDrive/Seed Grant Project"

RAW_HTML = os.path.join(BASE_DIR, "raw_html")
RAW_TXT = os.path.join(BASE_DIR, "raw_txt")

os.makedirs(RAW_HTML, exist_ok=True)
os.makedirs(RAW_TXT, exist_ok=True)

print("Folders ready")

Folders ready


In [ ]:
HEADERS = {
    "User-Agent": "James Ashford apwbd2005@gmail.com"
}

import time
import threading

LAST_REQUEST_TIME = 0

def safe_request(url, headers, min_interval=0.3, retries=5):
    global LAST_REQUEST_TIME

    for attempt in range(retries):

        # 🔥 FORCE SPACING (KEY FIX)
        now = time.time()
        elapsed = now - LAST_REQUEST_TIME

        if elapsed < min_interval:
            time.sleep(min_interval - elapsed)

        LAST_REQUEST_TIME = time.time()

        try:
            r = requests.get(url, headers=headers)

            if r.status_code == 200:
                return r

            elif r.status_code == 429:
                sleep_time = 60
                print(f"🚫 BLOCKED 429 → sleeping {sleep_time}s")
                time.sleep(sleep_time)

            else:
                print(f"❌ FAILED REQUEST: {url}")

        except:
            time.sleep(2)

    return None

## Submission JSON

In [ ]:
def get_submission_json(cik):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    r = safe_request(url, HEADERS)
    if r is None:
        return None
    return r.json()

# Define CIK list

In [ ]:
def get_index_file(year, quarter):
    url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/QTR{quarter}/master.idx"

    r = safe_request(url, HEADERS)
    if r is None:
        print(f"❌ Failed: {year} Q{quarter}")
        return None

    text = r.text

    # skip header (first ~11 lines)
    lines = text.splitlines()

    start_idx = 0
    for i, line in enumerate(lines):
        if line.startswith("CIK|"):
            start_idx = i + 1
            break

    data_lines = lines[start_idx:]

    return data_lines

In [ ]:
def parse_index_lines(lines):

    records = []

    for line in lines:
        parts = line.split("|")

        if len(parts) != 5:
            continue

        cik, name, form, date, path = parts

        # 🔥 FILTER HERE
        if form not in ["10-K", "10-K/A"]:
            continue

        accession = path.split("/")[-1].replace(".txt", "")

        records.append({
            "cik": cik.zfill(10),
            "company": name,
            "form": form,
            "filing_date": date,
            "accession": accession,
            "file_path": path
        })

    return pd.DataFrame(records)

In [ ]:
START_YEAR = 2009
END_YEAR = 2025

all_dfs = []

for year in range(START_YEAR, END_YEAR + 1):
    for q in range(1, 5):

        print(f"📦 {year} Q{q}")

        lines = get_index_file(year, q)

        if lines is None:
            continue

        df = parse_index_lines(lines)

        print(f"   → {len(df)} 10-K found")

        all_dfs.append(df)

        time.sleep(0.5)  # be nice to SEC

📦 2009 Q1
   → 6779 10-K found
📦 2009 Q2
   → 2586 10-K found
📦 2009 Q3
   → 1382 10-K found
📦 2009 Q4
   → 1412 10-K found
📦 2010 Q1
   → 6486 10-K found
📦 2010 Q2
   → 2417 10-K found
📦 2010 Q3
   → 1206 10-K found
📦 2010 Q4
   → 1269 10-K found
📦 2011 Q1
   → 6386 10-K found
📦 2011 Q2
   → 2129 10-K found
📦 2011 Q3
   → 1189 10-K found
📦 2011 Q4
   → 1131 10-K found
📦 2012 Q1
   → 6051 10-K found
📦 2012 Q2
   → 2051 10-K found
📦 2012 Q3
   → 1020 10-K found
📦 2012 Q4
   → 1111 10-K found
📦 2013 Q1
   → 5349 10-K found
📦 2013 Q2
   → 2449 10-K found
📦 2013 Q3
   → 1079 10-K found
📦 2013 Q4
   → 993 10-K found
📦 2014 Q1
   → 5936 10-K found
📦 2014 Q2
   → 1890 10-K found
📦 2014 Q3
   → 913 10-K found
📦 2014 Q4
   → 902 10-K found
📦 2015 Q1
   → 5998 10-K found
📦 2015 Q2
   → 1684 10-K found
📦 2015 Q3
   → 835 10-K found
📦 2015 Q4
   → 726 10-K found
📦 2016 Q1
   → 5734 10-K found
📦 2016 Q2
   → 1538 10-K found
📦 2016 Q3
   → 792 10-K found
📦 2016 Q4
   → 700 10-K found
📦 2017 Q1
   → 

In [ ]:
full_df = pd.concat(all_dfs, ignore_index=True)

# remove duplicates (VERY IMPORTANT)
full_df = full_df.drop_duplicates(subset=["accession"])

# sort
full_df["filing_date"] = pd.to_datetime(full_df["filing_date"])
full_df = full_df.sort_values("filing_date")

print("Total filings:", len(full_df))
print("Unique companies:", full_df["cik"].nunique())

Total filings: 149158
Unique companies: 18501


In [ ]:
# full dataset
full_path = os.path.join(BASE_DIR, "all_10k_filings_2009_present.csv")
full_df.to_csv(full_path, index=False)

# unique companies
cik_df = full_df[["cik"]].drop_duplicates()
cik_path = os.path.join(BASE_DIR, "valid_ciks_from_index.csv")
cik_df.to_csv(cik_path, index=False)

print("✅ Saved:")
print(full_path)
print(cik_path)

✅ Saved:
/content/drive/MyDrive/Seed Grant Project/all_10k_filings_2009_present.csv
/content/drive/MyDrive/Seed Grant Project/valid_ciks_from_index.csv


## Downloaders

In [ ]:
def download_primary_html(cik, accession, document):

    accession_clean = accession.replace("-", "")
    folder = os.path.join(RAW_HTML, cik)
    os.makedirs(folder, exist_ok=True)

    path = os.path.join(folder, f"{accession}.html")

    if os.path.exists(path):
        return True

    url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession_clean}/{document}"

    r = safe_request(url, HEADERS)
    if r is None:
        return False

    with open(path, "w", encoding="utf-8") as f:
        f.write(r.text)

    return True


def download_submission_txt(cik, accession):

    accession_clean = accession.replace("-", "")
    folder = os.path.join(RAW_TXT, cik)
    os.makedirs(folder, exist_ok=True)

    path = os.path.join(folder, f"{accession}.txt")

    if os.path.exists(path):
        return True

    url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession_clean}/{accession}.txt"

    r = safe_request(url, HEADERS)
    if r is None:
        return False

    with open(path, "w", encoding="utf-8") as f:
        f.write(r.text)

    return True

## Parallel Loop

In [ ]:
import pandas as pd
import os

FULL_PATH = os.path.join(BASE_DIR, "all_10k_filings_2009_present.csv")

full_df = pd.read_csv(FULL_PATH)

# 🔥 restore datetime (important)
full_df["filing_date"] = pd.to_datetime(full_df["filing_date"])

print("Loaded:", len(full_df))
full_df.head()

Loaded: 149158


,cik,company,form,filing_date,accession,file_path
0,1415813,ASIARIM CORP,10-K,2009-01-02,0001415812-09-000001,edgar/data/1415813/0001415812-09-000001.txt
1,788329,JOHNSON OUTDOORS INC,10-K,2009-01-02,0001042167-09-000004,edgar/data/788329/0001042167-09-000004.txt
2,1117171,CHINA BAK BATTERY INC,10-K/A,2009-01-05,0001144204-09-000201,edgar/data/1117171/0001144204-09-000201.txt
3,1356371,Nugget Resources Inc.,10-K,2009-01-05,0001176256-09-000004,edgar/data/1356371/0001176256-09-000004.txt
4,22444,COMMERCIAL METALS CO,10-K/A,2009-01-05,0000950134-09-000066,edgar/data/22444/0000950134-09-000066.txt


In [ ]:
full_df = pd.read_csv(FULL_PATH)

# 🔥 CRITICAL FIXES
full_df["cik"] = full_df["cik"].astype(str).str.zfill(10)
full_df["accession"] = full_df["accession"].astype(str)
full_df["file_path"] = full_df["file_path"].astype(str)
full_df["filing_date"] = pd.to_datetime(full_df["filing_date"])

print("Loaded:", len(full_df))

Loaded: 149158


In [ ]:
BATCH_ID = 3        # 🔥 change each run: 0,1,2,...
BATCH_SIZE = 10000

start = BATCH_ID * BATCH_SIZE
end = start + BATCH_SIZE

batch_df = full_df.iloc[start:end].copy()

print(f"Running batch {BATCH_ID}")
print(f"Range: {start} → {end}")
print(f"Batch size: {len(batch_df)}")

Running batch 3
Range: 30000 → 40000
Batch size: 10000


In [ ]:
DONE_PATH = os.path.join(BASE_DIR, f"done_accessions_batch_{BATCH_ID}.txt")

if os.path.exists(DONE_PATH):
    with open(DONE_PATH, "r") as f:
        done_accessions = set([line.strip() for line in f.readlines()])
    print("Loaded done:", len(done_accessions))
else:

    done_accessions = set()

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def process_filing(row):
    cik = row["cik"]
    accession = row["accession"]

    # 🔥 SKIP IF ALREADY DONE
    if accession in done_accessions:
        return None

    filing_date = row["filing_date"]
    file_path = row["file_path"]
    document = file_path.split("/")[-1]

    html_ok = download_primary_html(cik, accession, document)
    txt_ok = download_submission_txt(cik, accession)

    success = html_ok or txt_ok

    return {
        "cik": cik,
        "filing_date": filing_date,
        "accession": accession,
        "html": html_ok,
        "txt": txt_ok,
        "success": success
    }

In [ ]:
metadata_company = {}
metadata_filing = []
fail_log = []

MAX_WORKERS = 3
SAVE_EVERY = 400
LOG_EVERY = 100

done = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [executor.submit(process_filing, row) for _, row in batch_df.iterrows()]

    for future in tqdm(as_completed(futures), total=len(futures)):

        result = future.result()

        if result is None:
            continue

        cik = result["cik"]
        accession = result["accession"]
        success = result["success"]

        done += 1

        # ---- log ----
        metadata_filing.append(result)

        if not success:
            fail_log.append((cik, accession))

        # ---- company ----
        if cik not in metadata_company:
            metadata_company[cik] = {
                "cik": cik,
                "num_10k": 0,
                "any_downloaded": False
            }

        metadata_company[cik]["num_10k"] += 1

        if success:
            metadata_company[cik]["any_downloaded"] = True

        # 🔹 mark done
        done_accessions.add(accession)

        # 🔹 logging
        if done % LOG_EVERY == 0:
            print(f"[BATCH {BATCH_ID}] done={done}")

        # 🔹 checkpoint
        if done % SAVE_EVERY == 0:

            print("💾 Saving checkpoint...")

            pd.DataFrame(metadata_filing).to_csv(
                os.path.join(BASE_DIR, f"filing_batch_{BATCH_ID}.csv"),
                index=False
            )

            with open(DONE_PATH, "w") as f:
                for acc in done_accessions:
                    f.write(acc + "\n")

            print("✅ Saved")

  1%|          | 100/10000 [00:38<1:10:28,  2.34it/s]

[BATCH 3] done=100


  2%|▏         | 200/10000 [01:18<1:02:45,  2.60it/s]

[BATCH 3] done=200


  3%|▎         | 300/10000 [01:49<48:12,  3.35it/s]

...

[BATCH 3] done=9800


 99%|█████████▉| 9900/10000 [49:31<00:29,  3.40it/s]

[BATCH 3] done=9900


100%|██████████| 10000/10000 [49:50<00:00,  3.34it/s]

[BATCH 3] done=10000
💾 Saving checkpoint...
✅ Saved


In [ ]:
print("💾 Final batch save...")

# ---- Filing ----
filing_batch_path = os.path.join(BASE_DIR, f"filing_batch_{BATCH_ID}.csv")
pd.DataFrame(metadata_filing).to_csv(filing_batch_path, index=False)

# ---- Company ----
company_df = pd.DataFrame(metadata_company.values())
company_df["has_10k"] = True
company_df["note"] = "from_batch"

company_batch_path = os.path.join(BASE_DIR, f"company_batch_{BATCH_ID}.csv")
company_df.to_csv(company_batch_path, index=False)

# ---- Fail ----
fail_df = pd.DataFrame(fail_log, columns=["cik", "accession"])
fail_batch_path = os.path.join(BASE_DIR, f"failed_batch_{BATCH_ID}.csv")
fail_df.to_csv(fail_batch_path, index=False)

# ---- Done accessions ----
with open(DONE_PATH, "w") as f:
    for acc in done_accessions:
        f.write(acc + "\n")

print("✅ Batch final save complete")
print("Saved:")
print(filing_batch_path)
print(company_batch_path)
print(fail_batch_path)

💾 Final batch save...
✅ Batch final save complete
Saved:
/content/drive/MyDrive/Seed Grant Project/filing_batch_3.csv
/content/drive/MyDrive/Seed Grant Project/company_batch_3.csv
/content/drive/MyDrive/Seed Grant Project/failed_batch_3.csv


## Merge

In [ ]:
import glob
import pandas as pd
import os
from pandas.errors import EmptyDataError

batch_files = glob.glob(os.path.join(BASE_DIR, "filing_batch_*.csv"))

print("Found batch files:", len(batch_files))

dfs = []
bad_files = []

for f in batch_files:

    try:
        df = pd.read_csv(f)

        # skip truly empty dfs
        if len(df.columns) == 0:
            bad_files.append(f)
            continue

        dfs.append(df)

    except EmptyDataError:
        bad_files.append(f)

    except Exception as e:
        print(f"❌ Error reading {f}: {e}")
        bad_files.append(f)

print("Valid batch files:", len(dfs))
print("Bad/empty files:", len(bad_files))

if bad_files:
    print("\n⚠️ Problem files:")
    for b in bad_files:
        print(b)

# ---- MERGE ----
all_df = pd.concat(dfs, ignore_index=True)

# ---- CLEAN ----
all_df = all_df.drop_duplicates(subset=["cik", "accession"])

all_df["filing_date"] = pd.to_datetime(all_df["filing_date"])

# ---- SAVE ----
final_path = os.path.join(BASE_DIR, "filing_metadata.csv")

all_df.to_csv(final_path, index=False)

print("✅ Final filing metadata saved")
print("Total filings:", len(all_df))
print("Unique companies:", all_df["cik"].nunique())

Found batch files: 15
Valid batch files: 15
Bad/empty files: 0
✅ Final filing metadata saved
Total filings: 108545
Unique companies: 18018


In [ ]:
company_df = (
    all_df
    .groupby("cik")
    .agg(
        num_10k=("accession", "count"),
        any_downloaded=("success", "max")
    )
    .reset_index()
)

company_df["has_10k"] = True
company_df["note"] = "from_index"

company_path = os.path.join(BASE_DIR, "company_metadata.csv")

company_df.to_csv(company_path, index=False)

print("✅ Company metadata saved")
print("Companies:", len(company_df))

✅ Company metadata saved
Companies: 18018


In [ ]:
fail_files = glob.glob(os.path.join(BASE_DIR, "failed_batch_*.csv"))

fail_dfs = []

for f in fail_files:

    try:
        df = pd.read_csv(f)
        fail_dfs.append(df)

    except:
        print(f"Skipping bad fail file: {f}")

if fail_dfs:

    fail_df = pd.concat(fail_dfs, ignore_index=True)
    fail_df = fail_df.drop_duplicates()

    fail_path = os.path.join(BASE_DIR, "failed.csv")

    fail_df.to_csv(fail_path, index=False)

    print("⚠️ Final failed.csv saved")
    print("Failed filings:", len(fail_df))

⚠️ Final failed.csv saved
Failed filings: 0
